# Definição das Métricas Técnicas de Negócio

## Objetivo

Neste notebook são definidas as métricas utilizadas para avaliar o desempenho dos modelos de Machine Learning desenvolvidos para previsão de churn de clientes.

Além da justificativa para cada métrica, também será realizada a preparação do conjunto de dados que será utilizado nos experimentos dos próximos notebooks.

Ao final deste notebook estarão definidos:

- As métricas técnicas de avaliação;
- A métrica de negócio do projeto;
- O conjunto de treino e teste;
- A estratégia de avaliação dos modelos.

In [1]:
# ============================================================
# ETAPA 1 - IMPORTAÇÃO DAS BIBLIOTECAS
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

In [2]:
# ============================================================
# ETAPA 2 - CARREGAMENTO DA BASE DE DADOS
# ============================================================
# Localizando a raiz do projeto a partir do diretório atual do
# notebook e carregando a base armazenada em data/raw.
#
# O uso de um caminho relativo à raiz torna o notebook portátil,
# evitando dependência de caminhos específicos da máquina local.
# ============================================================

PROJECT_ROOT = Path.cwd().parents[2]

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset não encontrado em: {DATA_PATH.resolve()}"
    )

df = pd.read_csv(DATA_PATH)

print("Base carregada com sucesso.")
print(f"Quantidade de registros: {df.shape[0]:,}".replace(",", "."))
print(f"Quantidade de variáveis: {df.shape[1]}")
print(f"Arquivo utilizado: {DATA_PATH.name}")

display(df.head())

Base carregada com sucesso.
Quantidade de registros: 7.043
Quantidade de variáveis: 21
Arquivo utilizado: WA_Fn-UseC_-Telco-Customer-Churn.csv


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# ============================================================
# ETAPA 3 - PREPARAÇÃO DA VARIÁVEL ALVO
# ============================================================
# Validando e convertendo a variável alvo Churn para formato
# numérico, necessário para o treinamento e avaliação dos
# modelos de classificação.
#
# Mapeamento:
#   No  -> 0 (Permaneceu)
#   Yes -> 1 (Cancelou)
# ============================================================

print("Valores encontrados na variável alvo:")
display(df["Churn"].value_counts(dropna=False))

# Conversão da variável alvo
y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

# Validação da conversão
assert y.isna().sum() == 0, (
    "Foram encontrados valores inesperados na variável Churn."
)

print("\nDistribuição da variável alvo após conversão:")
display(
    y.value_counts()
     .rename(index={0: "Permaneceu (0)", 1: "Cancelou (1)"})
)

Valores encontrados na variável alvo:


Churn
No     5174
Yes    1869
Name: count, dtype: int64


Distribuição da variável alvo após conversão:


Churn
Permaneceu (0)    5174
Cancelou (1)      1869
Name: count, dtype: int64

In [4]:
# ============================================================
# ETAPA 3.1 - DEFINIÇÃO DAS VARIÁVEIS EXPLICATIVAS
# ============================================================
# Separando as variáveis utilizadas como entrada do modelo.
#
# customerID é removida por representar apenas um identificador
# único do cliente, sem significado preditivo direto.
#
# Churn é removida por ser a variável alvo que o modelo deverá
# aprender a prever.
# ============================================================

X = df.drop(columns=["customerID", "Churn"])

print(f"Quantidade de registros: {X.shape[0]:,}".replace(",", "."))
print(f"Quantidade de variáveis explicativas: {X.shape[1]}")

display(X.head())

Quantidade de registros: 7.043
Quantidade de variáveis explicativas: 19


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65


In [5]:
# ============================================================
# ETAPA 4 - DIVISÃO DOS DADOS EM TREINO E TESTE
# ============================================================
# Separando os dados em conjuntos de treino (80%) e teste (20%).
#
# A estratificação pela variável alvo (stratify=y) preserva
# aproximadamente a mesma proporção de clientes que permaneceram
# e cancelaram em ambos os conjuntos.
#
# O random_state garante a reprodutibilidade da divisão,
# permitindo reproduzir os mesmos conjuntos em execuções futuras.
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Dimensões dos conjuntos:")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test:  {y_test.shape}")

Dimensões dos conjuntos:
X_train: (5634, 19)
X_test:  (1409, 19)
y_train: (5634,)
y_test:  (1409,)


In [7]:
# ============================================================
# ETAPA 4.1 - VALIDAÇÃO DA ESTRATIFICAÇÃO
# ============================================================
# Comparando a distribuição percentual da variável alvo na base
# completa, no conjunto de treino e no conjunto de teste para
# verificar se a proporção das classes foi preservada.
# ============================================================

distribuicao_classes = pd.DataFrame({
    "Base Completa (%)": y.value_counts(normalize=True).sort_index() * 100,
    "Treino (%)": y_train.value_counts(normalize=True).sort_index() * 100,
    "Teste (%)": y_test.value_counts(normalize=True).sort_index() * 100
})

distribuicao_classes.index = [
    "Permaneceu (0)",
    "Cancelou (1)"
]

display(distribuicao_classes.round(2))

,Base Completa (%),Treino (%),Teste (%)
Permaneceu (0),73.46,73.46,73.46
Cancelou (1),26.54,26.54,26.54


### Conclusão da Divisão dos Dados

A base foi dividida em 80% para treinamento e 20% para teste, utilizando amostragem estratificada pela variável alvo.

A comparação entre as distribuições demonstra que a proporção de clientes que permaneceram e cancelaram foi preservada nos conjuntos de treino e teste, mantendo aproximadamente 73,4% de clientes sem churn e 26,5% de clientes com churn.

A utilização da estratificação é especialmente importante neste problema devido ao desbalanceamento moderado observado entre as classes, garantindo que ambos os conjuntos sejam representativos da distribuição original dos dados.

O conjunto de teste permanecerá isolado durante o treinamento e será utilizado posteriormente para avaliar a capacidade de generalização dos modelos.

# =================================================
# ETAPA 5 - Definição das Métricas Técnicas
# =================================================
A avaliação de um modelo de classificação de churn exige a utilização de métricas que considerem não apenas a quantidade total de previsões corretas, mas também os diferentes tipos de erro cometidos pelo modelo.

Neste projeto, a classe positiva é definida como:

- **0 — Permaneceu:** cliente que não apresentou churn.
- **1 — Cancelou:** cliente que apresentou churn.

A Análise Exploratória dos Dados identificou um desbalanceamento moderado entre as classes, com aproximadamente **73,46% dos clientes permanecendo** e **26,54% cancelando** o serviço.

Por esse motivo, a acurácia não será utilizada isoladamente como critério de seleção dos modelos.

As métricas técnicas adotadas serão:

- **Accuracy (Acurácia):** desempenho global das classificações;
- **Precision (Precisão):** confiabilidade das previsões de churn;
- **Recall (Sensibilidade):** capacidade de identificar clientes que realmente cancelarão;
- **F1-Score:** equilíbrio entre Precision e Recall;
- **ROC-AUC:** capacidade global de discriminação entre as classes;
- **PR-AUC:** desempenho na identificação da classe positiva, considerando o desbalanceamento existente.

A avaliação será realizada utilizando múltiplas métricas, pois cada uma representa uma perspectiva diferente do desempenho do modelo.

In [8]:
# ============================================================
# ETAPA 5.1 - ESTRUTURA DAS MÉTRICAS TÉCNICAS
# ============================================================
# Organizando as métricas que serão utilizadas para avaliar e
# comparar os modelos desenvolvidos ao longo do projeto.
# ============================================================

metricas_tecnicas = pd.DataFrame({
    "Métrica": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "ROC-AUC",
        "PR-AUC"
    ],
    "Objetivo": [
        "Avaliar a proporção total de previsões corretas",
        "Avaliar a confiabilidade das previsões de churn",
        "Identificar corretamente clientes que realmente cancelarão",
        "Equilibrar Precision e Recall",
        "Avaliar a capacidade global de discriminação entre as classes",
        "Avaliar o desempenho sobre a classe positiva em cenário desbalanceado"
    ],
    "Prioridade": [
        "Secundária",
        "Secundária",
        "Alta",
        "Alta",
        "Alta",
        "Principal"
    ]
})

display(metricas_tecnicas)

,Métrica,Objetivo,Prioridade
0,Accuracy,Avaliar a proporção total de previsões corretas,Secundária
1,Precision,Avaliar a confiabilidade das previsões de churn,Secundária
2,Recall,Identificar corretamente clientes que realment...,Alta
3,F1-Score,Equilibrar Precision e Recall,Alta
4,ROC-AUC,Avaliar a capacidade global de discriminação e...,Alta
5,PR-AUC,Avaliar o desempenho sobre a classe positiva e...,Principal


## ETAPA 5.2 - Definição da Métrica Técnica Principal

A métrica **PR-AUC (Area Under the Precision-Recall Curve)** será utilizada como **principal referência técnica para comparação e seleção dos modelos** desenvolvidos neste projeto.

Essa escolha considera o desbalanceamento observado na variável alvo, em que aproximadamente **26,54% dos clientes pertencem à classe positiva (Churn = 1)**. Nesse contexto, a PR-AUC permite avaliar de forma mais adequada o desempenho do modelo na identificação dos clientes com risco de cancelamento, analisando conjuntamente o comportamento de **Precision** e **Recall** em diferentes limiares de decisão.

Embora a **PR-AUC seja a métrica técnica principal**, a avaliação dos modelos não será baseada exclusivamente em um único indicador.

Para garantir uma análise comparativa mais completa, também serão reportadas as seguintes métricas:

- **F1-Score:** utilizada para avaliar o equilíbrio entre Precision e Recall;
- **ROC-AUC:** utilizada para avaliar a capacidade global de discriminação entre clientes que permanecem e clientes que cancelam;
- **Recall:** utilizada para medir a capacidade de identificar corretamente clientes que efetivamente apresentam churn;
- **Precision:** utilizada para avaliar a confiabilidade das previsões positivas;
- **Accuracy:** utilizada como métrica complementar de desempenho geral, sem ser considerada isoladamente para seleção do modelo.

Dessa forma, a seleção do modelo campeão considerará prioritariamente a **PR-AUC**, acompanhada da análise de **F1-Score, ROC-AUC e Recall**, além das demais métricas complementares e dos impactos de negócio associados aos Falsos Positivos e Falsos Negativos.

Esse protocolo será aplicado de forma consistente aos modelos avaliados ao longo do projeto, permitindo uma comparação justa entre o **baseline de Regressão Logística**, os modelos baseados em **Árvores/Ensembles** e a **Rede Neural MLPClassifier**.

In [9]:
# ============================================================
# ETAPA 5.3 - REFERÊNCIA BASE PARA A PR-AUC
# ============================================================
# A prevalência da classe positiva fornece uma referência
# natural para interpretar o desempenho futuro dos modelos
# na métrica PR-AUC / Average Precision.
# ============================================================

prevalencia_churn = y_train.mean()

print(
    f"Prevalência de churn no conjunto de treino: "
    f"{prevalencia_churn:.2%}"
)

print(
    f"Referência aproximada para PR-AUC de um classificador "
    f"aleatório: {prevalencia_churn:.4f}"
)

Prevalência de churn no conjunto de treino: 26.54%
Referência aproximada para PR-AUC de um classificador aleatório: 0.2654


# ETAPA 6 - Interpretação dos Erros de Classificação

Além das métricas agregadas, é fundamental compreender os diferentes tipos de acertos e erros que podem ser produzidos pelos modelos de classificação.

Neste projeto, considera-se **Churn = 1** como a classe positiva.

A matriz de confusão será interpretada da seguinte forma:

| Resultado | Interpretação no contexto de churn |
|---|---|
| **Verdadeiro Negativo (TN)** | O modelo prevê que o cliente permanecerá e ele realmente permanece. |
| **Falso Positivo (FP)** | O modelo prevê churn, mas o cliente permaneceria. |
| **Falso Negativo (FN)** | O modelo prevê permanência, mas o cliente realmente cancela. |
| **Verdadeiro Positivo (TP)** | O modelo prevê churn e o cliente realmente cancela. |

Os dois tipos de erro possuem impactos distintos para o negócio.

Um **Falso Positivo (FP)** pode gerar uma ação de retenção desnecessária, como a concessão de descontos, benefícios ou contato comercial para um cliente que não pretendia cancelar.

Um **Falso Negativo (FN)** representa um cliente com risco real de churn que não foi identificado pelo modelo. Nesse cenário, a empresa pode perder a oportunidade de realizar uma ação preventiva de retenção.

Por esse motivo, os Falsos Negativos possuem especial relevância neste projeto. Entretanto, maximizar o Recall indiscriminadamente também pode aumentar o número de Falsos Positivos e, consequentemente, elevar os custos das campanhas de retenção.

A avaliação dos modelos deverá, portanto, buscar um equilíbrio entre a capacidade de identificar clientes em risco e o custo associado às intervenções realizadas.

In [10]:
# ============================================================
# ETAPA 6.1 - TIPOS DE RESULTADOS DA CLASSIFICAÇÃO
# ============================================================
# Estruturando a interpretação dos possíveis resultados de uma
# matriz de confusão no contexto do problema de churn.
# ============================================================

resultados_classificacao = pd.DataFrame({
    "Resultado": [
        "Verdadeiro Negativo (TN)",
        "Falso Positivo (FP)",
        "Falso Negativo (FN)",
        "Verdadeiro Positivo (TP)"
    ],
    "Predição": [
        "Permaneceu (0)",
        "Cancelou (1)",
        "Permaneceu (0)",
        "Cancelou (1)"
    ],
    "Real": [
        "Permaneceu (0)",
        "Permaneceu (0)",
        "Cancelou (1)",
        "Cancelou (1)"
    ],
    "Impacto no Negócio": [
        "Classificação correta de cliente sem churn",
        "Possível custo de retenção desnecessário",
        "Cliente em risco não identificado e possível perda de receita",
        "Cliente em risco corretamente identificado para possível intervenção"
    ]
})

display(resultados_classificacao)

,Resultado,Predição,Real,Impacto no Negócio
0,Verdadeiro Negativo (TN),Permaneceu (0),Permaneceu (0),Classificação correta de cliente sem churn
1,Falso Positivo (FP),Cancelou (1),Permaneceu (0),Possível custo de retenção desnecessário
2,Falso Negativo (FN),Permaneceu (0),Cancelou (1),Cliente em risco não identificado e possível p...
3,Verdadeiro Positivo (TP),Cancelou (1),Cancelou (1),Cliente em risco corretamente identificado par...


## ETAPA 6.2 - Relação entre os Erros e as Métricas

As métricas selecionadas refletem diferentes consequências dos erros de classificação no contexto da previsão de churn:

- **Recall:** mede a proporção de clientes que realmente apresentaram churn e foram corretamente identificados pelo modelo. Um Recall baixo está associado a uma maior proporção de Falsos Negativos, representando clientes em risco que podem deixar de receber uma ação preventiva de retenção.

- **Precision:** mede, entre os clientes classificados como risco de churn, a proporção daqueles que realmente cancelaram. Uma Precision baixa indica maior ocorrência proporcional de Falsos Positivos, podendo resultar em ações de retenção desnecessárias e aumento dos custos operacionais.

- **F1-Score:** representa a média harmônica entre Precision e Recall, permitindo avaliar o equilíbrio entre a identificação dos clientes em risco e a redução de intervenções desnecessárias.

- **ROC-AUC:** avalia a capacidade geral do modelo de discriminar clientes com e sem churn ao longo de diferentes limiares de classificação.

- **PR-AUC:** avalia conjuntamente Precision e Recall em diferentes limiares de decisão, sendo especialmente relevante neste projeto devido ao foco na classe positiva (Churn = 1) e ao desbalanceamento observado entre as classes.

Dessa forma, a seleção do modelo final não será baseada em uma única métrica. A **PR-AUC será utilizada como principal referência técnica**, enquanto Recall, F1-Score e ROC-AUC fornecerão perspectivas complementares sobre o desempenho do modelo.

Essa avaliação multidimensional permitirá considerar tanto a capacidade preditiva dos modelos quanto as consequências dos Falsos Positivos e Falsos Negativos para as futuras estratégias de retenção.

# ETAPA 7 - Definição da Métrica de Negócio

Além das métricas técnicas, a avaliação de um modelo de churn deve considerar o impacto econômico das decisões geradas a partir de suas previsões.

Neste projeto, a métrica de negócio será baseada no **valor econômico líquido potencial das ações de retenção**, considerando a identificação de clientes com risco de churn e os custos associados às intervenções.

O conjunto de dados utilizado não fornece diretamente informações como:

- custo de uma campanha de retenção;
- valor financeiro futuro de cada cliente;
- probabilidade real de sucesso de uma ação de retenção;
- custo efetivo decorrente da perda de um cliente.

Por esse motivo, esses valores não serão tratados como fatos observados. Será utilizada uma estrutura parametrizável de simulação, permitindo avaliar diferentes cenários de negócio sem introduzir premissas financeiras como se fossem dados reais.

A lógica econômica considera principalmente:

- **Verdadeiros Positivos (TP):** clientes com churn corretamente identificados, que podem receber uma ação preventiva;
- **Falsos Positivos (FP):** clientes que receberiam uma ação de retenção desnecessariamente;
- **Falsos Negativos (FN):** clientes com churn não identificados, representando oportunidades de retenção perdidas.

Essa abordagem permite conectar o desempenho técnico do modelo ao impacto potencial para o negócio.

## ETAPA 7.1 - Custo de Churn Evitado

Para avaliar o impacto econômico potencial do modelo, será utilizada uma estimativa de **Valor Líquido de Retenção**, baseada na seguinte lógica:

**Valor Líquido = Valor do churn potencialmente evitado − Custo das intervenções**

De forma parametrizada:

**Valor Líquido = (TP × taxa de sucesso da retenção × valor do cliente) − ((TP + FP) × custo da intervenção)**

Onde:

- **TP:** clientes com churn corretamente identificados;
- **FP:** clientes sem churn classificados incorretamente como risco;
- **taxa de sucesso da retenção:** proporção estimada de clientes em risco que permaneceriam após uma intervenção;
- **valor do cliente:** valor econômico estimado preservado quando um churn é evitado;
- **custo da intervenção:** custo associado à ação de retenção realizada para cada cliente sinalizado pelo modelo.

Os **Falsos Negativos (FN)** não aparecem diretamente na fórmula de valor realizado, pois representam oportunidades que o modelo deixou de identificar. Entretanto, serão acompanhados por meio do Recall e poderão ser utilizados para estimar o valor potencial perdido.

Como o dataset não fornece os parâmetros financeiros necessários, a métrica será implementada como uma função parametrizável. Os valores utilizados posteriormente deverão ser explicitamente identificados como premissas de simulação e não como informações observadas na base.

In [11]:
# ============================================================
# ETAPA 7.2 - FUNÇÃO DA MÉTRICA DE NEGÓCIO
# ============================================================
# Função parametrizável para estimar o valor econômico líquido
# potencial de uma estratégia de retenção baseada nas previsões
# do modelo.
#
# Os parâmetros financeiros não estão disponíveis no dataset
# e deverão ser definidos posteriormente como premissas de
# cenário, sem serem tratados como valores reais observados.
# ============================================================

def calcular_valor_retencao(
    tp,
    fp,
    fn,
    valor_cliente,
    custo_intervencao,
    taxa_sucesso_retencao
):
    """
    Calcula indicadores econômicos potenciais de uma estratégia
    de retenção baseada nas previsões de churn.

    Parâmetros
    ----------
    tp : int
        Quantidade de verdadeiros positivos.

    fp : int
        Quantidade de falsos positivos.

    fn : int
        Quantidade de falsos negativos.

    valor_cliente : float
        Valor econômico estimado preservado por churn evitado.

    custo_intervencao : float
        Custo da ação de retenção por cliente abordado.

    taxa_sucesso_retencao : float
        Probabilidade estimada de sucesso da ação de retenção,
        representada por um valor entre 0 e 1.

    Retorna
    -------
    dict
        Indicadores econômicos estimados da estratégia.
    """

    if not 0 <= taxa_sucesso_retencao <= 1:
        raise ValueError(
            "A taxa de sucesso da retenção deve estar entre 0 e 1."
        )

    clientes_abordados = tp + fp

    churns_potencialmente_evitados = (
        tp * taxa_sucesso_retencao
    )

    valor_churn_evitado = (
        churns_potencialmente_evitados * valor_cliente
    )

    custo_total_intervencoes = (
        clientes_abordados * custo_intervencao
    )

    valor_liquido_estimado = (
        valor_churn_evitado - custo_total_intervencoes
    )

    valor_potencial_perdido_fn = (
        fn * taxa_sucesso_retencao * valor_cliente
    )

    return {
        "clientes_abordados": clientes_abordados,
        "churns_potencialmente_evitados": churns_potencialmente_evitados,
        "valor_churn_evitado": valor_churn_evitado,
        "custo_total_intervencoes": custo_total_intervencoes,
        "valor_liquido_estimado": valor_liquido_estimado,
        "valor_potencial_perdido_fn": valor_potencial_perdido_fn
    }

In [12]:
# ============================================================
# ETAPA 7.3 - INDICADORES DE NEGÓCIO
# ============================================================
# Estruturando os indicadores que serão utilizados para conectar
# o desempenho dos modelos ao impacto potencial no negócio.
# ============================================================

metricas_negocio = pd.DataFrame({
    "Indicador": [
        "Clientes abordados",
        "Churns potencialmente evitados",
        "Valor do churn evitado",
        "Custo total das intervenções",
        "Valor líquido estimado",
        "Valor potencial perdido por FN"
    ],
    "Descrição": [
        "Quantidade de clientes sinalizados para ação de retenção",
        "Estimativa de clientes retidos entre os churns corretamente identificados",
        "Valor econômico potencial preservado pela retenção",
        "Custo total das ações realizadas nos clientes sinalizados",
        "Benefício econômico estimado após descontar os custos das intervenções",
        "Valor potencial associado aos clientes com churn não identificados"
    ]
})

display(metricas_negocio)

,Indicador,Descrição
0,Clientes abordados,Quantidade de clientes sinalizados para ação d...
1,Churns potencialmente evitados,Estimativa de clientes retidos entre os churns...
2,Valor do churn evitado,Valor econômico potencial preservado pela rete...
3,Custo total das intervenções,Custo total das ações realizadas nos clientes ...
4,Valor líquido estimado,Benefício econômico estimado após descontar os...
5,Valor potencial perdido por FN,Valor potencial associado aos clientes com chu...


### Conclusão da Métrica de Negócio

A avaliação de negócio foi estruturada de forma parametrizável, permitindo estimar o impacto econômico potencial das previsões sem assumir valores financeiros inexistentes no dataset.

A principal métrica econômica será o **Valor Líquido Estimado da Retenção**, calculado a partir do valor potencialmente preservado pelos churns evitados menos o custo das intervenções realizadas.

Além disso, serão acompanhados o número de clientes abordados, os churns potencialmente evitados e o valor potencial associado aos Falsos Negativos.

Essa abordagem permite que diferentes modelos e limiares de decisão sejam comparados não apenas por seu desempenho estatístico, mas também pelo impacto potencial de suas decisões sobre o negócio.

Os valores financeiros utilizados em simulações futuras serão tratados explicitamente como premissas de cenário e poderão ser substituídos por dados reais caso essas informações estejam disponíveis em um ambiente de produção.

# ETAPA 8 - Critérios para Comparação e Seleção dos Modelos

Para garantir uma comparação consistente e reprodutível, todos os modelos desenvolvidos neste projeto serão avaliados utilizando o mesmo conjunto de teste e o mesmo protocolo de avaliação.

O **DummyClassifier** será utilizado como baseline mínimo de referência. Seu objetivo não é fornecer uma solução preditiva competitiva, mas estabelecer um desempenho básico que deverá ser superado pelos modelos treinados posteriormente.

A comparação entre os modelos seguirá os seguintes critérios:

1. **PR-AUC como principal referência técnica**, devido ao foco na identificação da classe positiva (Churn = 1) e ao desbalanceamento observado entre as classes.

2. **Recall como indicador crítico complementar**, pois representa a capacidade de identificar clientes que realmente apresentam churn e reduz a ocorrência de Falsos Negativos.

3. **F1-Score para avaliar o equilíbrio entre Precision e Recall**, evitando favorecer modelos que obtenham bom desempenho em apenas uma dessas dimensões.

4. **ROC-AUC como medida complementar da capacidade discriminatória global** do modelo entre clientes com e sem churn.

5. **Precision para avaliar a eficiência das ações de retenção**, considerando que Falsos Positivos podem gerar intervenções desnecessárias.

6. **Accuracy como métrica descritiva complementar**, sem utilização isolada para seleção do melhor modelo devido ao desbalanceamento entre as classes.

7. **Impacto potencial no negócio**, avaliado por meio dos indicadores econômicos definidos anteriormente, quando estiverem disponíveis previsões reais e premissas de cenário.

O modelo final não será escolhido exclusivamente pela maior pontuação em uma única métrica. A decisão deverá considerar conjuntamente desempenho técnico, tipos de erro, capacidade de generalização e impacto potencial para o negócio.

In [ ]:
# ============================================================
# ETAPA 8.1 - PROTOCOLO DE AVALIAÇÃO DOS MODELOS
# ============================================================
# Formalizando os critérios que deverão ser aplicados de maneira
# consistente a todos os modelos avaliados no projeto.
# ============================================================

protocolo_avaliacao = pd.DataFrame({
    "Critério": [
        "Conjunto de avaliação",
        "Métrica técnica principal",
        "Métricas técnicas complementares",
        "Baseline oficial",
        "Classe positiva",
        "Análise de erros",
        "Avaliação de negócio",
        "Reprodutibilidade"
    ],
    "Definição": [
        "Mesmo conjunto de teste estratificado para todos os modelos",
        "PR-AUC",
        "Recall, F1-Score, ROC-AUC, Precision e Accuracy",
        "Regressão Logistica (Scikit-Learn)",
        "Churn = 1 (Cancelou)",
        "Matriz de confusão com análise de FP e FN",
        "Valor líquido estimado e indicadores de retenção",
        "random_state=42 e registro dos experimentos"
    ]
})

display(protocolo_avaliacao)

,Critério,Definição
0,Conjunto de avaliação,Mesmo conjunto de teste estratificado para tod...
1,Métrica técnica principal,PR-AUC
2,Métricas técnicas complementares,"Recall, F1-Score, ROC-AUC, Precision e Accuracy"
3,Baseline mínimo,DummyClassifier
4,Classe positiva,Churn = 1 (Cancelou)
5,Análise de erros,Matriz de confusão com análise de FP e FN
6,Avaliação de negócio,Valor líquido estimado e indicadores de retenção
7,Reprodutibilidade,random_state=42 e registro dos experimentos


In [25]:
# ============================================================
# ETAPA 8.2 - FUNÇÃO PADRONIZADA DE AVALIAÇÃO TÉCNICA
# ============================================================
# Criando uma função única para calcular as métricas técnicas
# dos modelos, garantindo consistência entre os experimentos.
#
# A função receberá:
# - valores reais;
# - classes previstas;
# - probabilidades da classe positiva.
# ============================================================

def avaliar_modelo(y_true, y_pred, y_proba):
    """
    Calcula as principais métricas técnicas utilizadas no projeto.

    Parâmetros
    ----------
    y_true : array-like
        Valores reais da variável alvo.

    y_pred : array-like
        Classes previstas pelo modelo.

    y_proba : array-like
        Probabilidades previstas para a classe positiva (Churn = 1).

    Retorna
    -------
    dict
        Dicionário contendo as métricas de avaliação.
    """

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "F1-Score": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_true,
            y_proba
        ),
        "PR-AUC": average_precision_score(
            y_true,
            y_proba
        )
    }

In [14]:
# ============================================================
# ETAPA 9 - RESUMO DAS DECISÕES DE AVALIAÇÃO
# ============================================================
# Consolidando as principais decisões metodológicas definidas
# para a avaliação dos modelos de previsão de churn.
# ============================================================

resumo_avaliacao = pd.DataFrame({
    "Elemento": [
        "Classe positiva",
        "Divisão dos dados",
        "Estratificação",
        "Métrica principal",
        "Métricas complementares",
        "Baseline inicial",
        "Erro crítico",
        "Métrica de negócio",
        "Controle de reprodutibilidade"
    ],
    "Definição": [
        "Churn = 1 (Cancelou)",
        "80% treino / 20% teste",
        "Preservação da proporção de Churn",
        "PR-AUC",
        "Recall, F1-Score, ROC-AUC, Precision e Accuracy",
        "DummyClassifier",
        "Falso Negativo (FN)",
        "Valor Líquido Estimado da Retenção",
        "random_state = 42"
    ]
})

display(resumo_avaliacao)

,Elemento,Definição
0,Classe positiva,Churn = 1 (Cancelou)
1,Divisão dos dados,80% treino / 20% teste
2,Estratificação,Preservação da proporção de Churn
3,Métrica principal,PR-AUC
4,Métricas complementares,"Recall, F1-Score, ROC-AUC, Precision e Accuracy"
5,Baseline inicial,DummyClassifier
6,Erro crítico,Falso Negativo (FN)
7,Métrica de negócio,Valor Líquido Estimado da Retenção
8,Controle de reprodutibilidade,random_state = 42


# ETAPA 10 - Conclusão

Neste notebook foi definida a estratégia de avaliação técnica e de negócio que será utilizada durante os experimentos de previsão de churn.

A base foi preparada com a separação da variável alvo e das variáveis explicativas, seguida pela divisão estratificada em **80% para treinamento e 20% para teste**. A estratificação preservou a distribuição original das classes, mantendo aproximadamente **73,46% de clientes sem churn e 26,54% de clientes com churn** em ambos os conjuntos.

Devido ao desbalanceamento observado e ao foco na identificação dos clientes com maior risco de cancelamento, a **PR-AUC foi definida como principal métrica técnica de comparação**. Recall, F1-Score, ROC-AUC, Precision e Accuracy serão utilizadas como métricas complementares, permitindo uma avaliação multidimensional do desempenho dos modelos.

Também foram formalizados os impactos dos diferentes tipos de erro. Os **Falsos Negativos (FN)** possuem especial relevância por representarem clientes com risco real de churn que não foram identificados pelo modelo, enquanto os **Falsos Positivos (FP)** podem gerar custos decorrentes de intervenções de retenção desnecessárias.

Sob a perspectiva de negócio, foi definida uma estrutura parametrizável para estimar o **Valor Líquido da Retenção**, considerando churns potencialmente evitados, custos de intervenção e oportunidades perdidas. Como o dataset não fornece parâmetros financeiros reais, quaisquer valores utilizados futuramente serão tratados explicitamente como premissas de simulação.

Por fim, foi estabelecido um protocolo padronizado e reprodutível para comparação dos modelos. O **DummyClassifier será utilizado como baseline mínimo**, seguido pela Regressão Logística e por modelos posteriores, todos avaliados utilizando o mesmo conjunto de teste e os mesmos critérios.

Com essa estrutura, os próximos experimentos poderão ser comparados de forma consistente tanto sob a perspectiva estatística quanto sob a perspectiva de impacto potencial para o negócio.

A Regressão Logística será utilizada como baseline oficial do projeto. Nas etapas seguintes, seu desempenho será comparado aos modelos baseados em Árvores/Ensembles e ao MLPClassifier do Scikit-Learn.

A comparação utilizará a PR-AUC como principal referência técnica, acompanhada por F1-Score, ROC-AUC, Recall, Precision e Accuracy, além da análise dos impactos de negócio associados às previsões.

Essa abordagem permitirá selecionar o modelo campeão de forma consistente, considerando não apenas o desempenho estatístico, mas também sua capacidade de identificar clientes com risco de churn e apoiar decisões de retenção.